# A combustion instability with no prescribed flame response

Every self-excited case in this collection so far carries at least one *dynamic source* — a flame or fuel-flow transfer function attached to an element.
This notebook shows that, under the right conditions, Nefes can model instabilities without any explicit specification of a dynamic source.
A steady fuel injector, a mixing duct, and a flame with no response model of its own are enough: the feedback that a transfer function normally *prescribes* is instead *assembled* from quasi-steady components and the convective transport of the duct.

The mechanism is the classical **equivalence-ratio (fuel-transport) coupling** of lean premixed combustors [1–3].
An acoustic fluctuation at the injector cannot change the fuel flow (the fuel line is stiff), but it does change the *air* flow, so it modulates the mixture strength of the parcel leaving the injector; the parcel convects to the flame and is burned into an unsteady heat release, delayed by the transport time.
Here that chain lives entirely in the linearized mean-flow physics — the base Jacobian and the convected composition wave — so the instability appears in the *passive* operator, with the source block empty.

The sibling notebook `equivalence_ratio_instability.ipynb` drives the same convective chain from the other end, with a prescribed fuel-flow response as the only dynamic element; here the fuel flow is constant and nothing is prescribed.

The plan for this notebook is: derive the emergent response, verify each link of the chain quantitatively against the derivation, exhibit the certified unstable modes, attribute them to the composition wave by switching individual convected families off, locate where the acoustic energy enters, and map the stability bands in the transport lag.

## The rig

A straight chain of seven elements, all of them steady or static:

```
mass-flow inlet ── approach duct ── fuel injector ── mixing duct ── flame ── hot duct ── choked-nozzle outlet
     (air)           0.5 m         (H2, steady)     0.15 m       (static)    0.5 m        (compact)
```

* The **inlet** fixes the air mass flow (acoustically stiff, $\widehat{\dot m} = 0$ at the boundary).
* The **injector** is a `mass_source`: it adds a constant $0.01\ \mathrm{kg/s}$ of hydrogen with no dynamic response of any kind.
* The **flame** is a static `equilibrium_flame`: it burns whatever mixture arrives, to chemical equilibrium, quasi-steadily.
* The **choked-nozzle outlet** is how a real combustor sets its pressure; it reflects almost fully, and it is where an arriving hot spot is accelerated and partly converted back into sound (the indirect-noise transduction [4, 6]).

No element carries a `dynamic_source`, and no boundary carries a prescribed response: the perturbation operator's source block is empty.
Whatever the spectrum shows is inherited linearized mean-flow physics.

## The mechanism, derived

Three steps take a fluctuation around the loop; we follow a fluid parcel.
Throughout, station $A$ is the injection plane, station $B$ the flame, and $\omega = 2\pi f$ the angular frequency of a time-harmonic perturbation $X'(t) = \Re\{\widehat{X}e^{\mathrm{i}\omega t}\}$.
The conventional mixture variable is the **equivalence ratio** $\phi = (\dot m_f/\dot m_a)/\mathrm{FAR}_{\mathrm{st}}$; its fluctuation $\phi'$ is what the literature tracks [2, 3].
Nefes transports the fuel-stream mixture fraction $\xi = \dot m_f/\dot m$ (with $\dot m = \dot m_a + \dot m_f$), which maps one-to-one onto $\phi$ by $\phi = \xi\big/\big((1-\xi)\,\mathrm{FAR}_{\mathrm{st}}\big)$; the two descriptions are interchangeable at first order.

### 1. Generation at the injector

The injector delivers a constant fuel flow $\overline{\dot m}_f$, so the mixture strength is at the mercy of the air flow.
Linearizing $\phi$ with $\dot m_f$ held fixed yields the classical stiff-fuel relation [2]:

$$
\frac{\widehat{\phi}_A}{\overline{\phi}} \;=\; -\,\frac{\widehat{\dot m}_{a,A}}{\overline{\dot m}_a}.
$$

A positive air-flow fluctuation produces a **leaner** parcel: the fuel flux is fixed, and what fluctuates is the diluent.
(The equivalent statement in mixture fraction is $\widehat{\xi}_A/\overline{\xi} = -\widehat{\dot m}_A/\overline{\dot m}$; for a lean mean, $\overline{\xi}\ll 1$ and $\overline{\dot m}_a\approx\overline{\dot m}$, so the two relative amplitudes nearly coincide.)
The air-stream fraction moves equally and oppositely — the stream fractions sum to one — so the wave trades fuel for air at constant total mass and carries no pressure or mass-flow signature of its own: it convects **silently**.

### 2. Transport down the mixing duct

The inhomogeneity is a transported scalar: it rides the mean flow and reaches the flame after the transit time $\tau = L_m/\overline{u}$, so

$$
\widehat{\phi}_B \;=\; \widehat{\phi}_A\, e^{-\mathrm{i}\omega\tau},
$$

where $L_m$ is the mixing-duct length and $\overline{u}$ the mean velocity in it.
For a uniform duct the fluctuation of the transit time itself enters only at second order; the sharp delay above is the complete first-order statement.

### 3. Conversion at the flame

A quasi-steady flame burns the fuel it is fed, completely and without memory, so its heat release is the arriving fuel flux times the heating value, $Q = \dot m_f\,\Delta h = \dot m\,\xi\,\Delta h$.
Linearizing yields

$$
\frac{\widehat{Q}}{\overline{Q}} \;=\; \frac{\widehat{\dot m}_B}{\overline{\dot m}} \;+\; \frac{\widehat{\xi}_B}{\overline{\xi}},
$$

a local, instantaneous part (more mass flux of the previously mixed composition) plus the delayed dilution imprinted at the injector.
Without assuming compactness, the same linearization plus the transported generation relation $\widehat{\xi}_B = -\overline{\xi}\,(\widehat{\dot m}_A/\overline{\dot m})\,e^{-\mathrm{i}\omega\tau}$ gives the exact **emergent flame transfer function** referenced to the injector face:

$$
\frac{\widehat{Q}/\overline{Q}}{\widehat{\dot m}_A/\overline{\dot m}} \;=\; \frac{\widehat{\dot m}_B}{\widehat{\dot m}_A} - e^{-\mathrm{i}\omega\tau}.
$$

When the injector-to-flame section is acoustically compact, $\omega L_m/\overline{c} \ll 1$, one has $\widehat{\dot m}_B\approx\widehat{\dot m}_A$ and this collapses to the classical shape $1 - e^{-\mathrm{i}\omega\tau}$ (gain $2\,|\sin(\omega\tau/2)|$, phase sweeping with frequency) — although nothing dynamic was prescribed.
Its zero-frequency limit vanishes, as it must: steadily adding air to a fixed fuel flow changes $\phi$ but not the total power $\overline{\dot m}_f\,\Delta h$.
In the rig below the mixing duct is not compact across the plotted band, so the notebook checks the exact form and overlays the compact idealization for reference.

### Two ways the heat release feeds the sound field

1. **Direct (Rayleigh) closure.** The compact flame is an acoustic monopole; the mode grows when the delayed heat release keeps a component in phase with the local pressure, which happens in periodic bands of $\omega\tau$.
2. **Indirect (accelerated-spot) closure.** Burning a richer parcel makes a *hotter* parcel — the burnt temperature rides the arriving $\phi$ through the equilibrium state $T_b(\phi)$ — and the hot spot is silent until it is accelerated, above all through the choked exit, where it partly converts to sound that travels back upstream [4, 6].

Both closures draw their phase from the same convective lag, and neither needs a prescribed response.
The rig below realizes the second one dominantly, and the notebook measures that.

### Liquid fuel and evaporation (open)

In many combustors the fuel is injected as liquid and must vaporize before it reaches the flame.
Near the front one then has a mix of vaporized and still-liquid fuel, and a rise in local heat release can raise the local evaporation rate — an extra, heat-release-dependent source of $\phi'$ that this gaseous model does not carry.
A possible route inside the present framework is to treat evaporation as a second, lagged mass source of vapor fuel on the mixing duct (or a dynamic source on the injector) whose gain couples to a flame-side reference: that would add a parallel $\phi'$ path beside the stiff-dilution path above, without changing the mean-flow topology.
That extension is left for a follow-up; the notebook stays with gaseous H₂ injection.


In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import nefes
from nefes.elements import catalog as cat
from nefes.perturbation import PerturbationBC
from nefes.plotting import use_nefes_theme, COLORWAY

# Use the bundled theme instead of plotly's default.
use_nefes_theme()

AIR = {"O2": 0.21, "N2": 0.79}
FAR_ST = 0.0292  # stoichiometric fuel/air mass ratio for H2–air
AREA = 0.02  # duct cross-section [m^2]
MDOT_AIR, MDOT_FUEL = 0.4, 0.010  # air / H2 [kg/s] -> phi ~ 0.86
MDOT = MDOT_AIR + MDOT_FUEL
XI_FUEL = MDOT_FUEL / MDOT  # mean fuel-stream mixture fraction
PHI = (MDOT_FUEL / MDOT_AIR) / FAR_ST  # mean equivalence ratio
LA, LM, LH = 0.5, 0.15, 0.5  # approach / mixing / hot lengths [m]
A_THROAT = 0.005  # choked-nozzle throat [m^2]

# Wave-vector columns: (f, g, h), then stream fractions in feed order (air, fuel), then burnt marker.
W_ENTROPY, W_XI_AIR, W_XI_FUEL = 2, 3, 4

# Eigensolver settings used throughout (band, and a dense counting contour: the
# reacting operator carries a ladder of convective modes the certificate must resolve).
EIG = dict(freq_band=(60.0, 500.0), growth_band=(-150.0, 120.0), n_nodes=512, n_probe=20)

# Base network: seven-element chain, no dynamic source on any element.
net = nefes.Network(
    nefes.equilibrium(),
    nodes=[
        cat.mass_flow_inlet(MDOT_AIR, 300.0, composition=AIR, basis="mole", name="inlet"),
        cat.duct(LA, name="approach"),
        cat.mass_source(MDOT_FUEL, 300.0, composition={"H2": 1.0}, basis="mole", name="injector"),
        cat.duct(LM, name="mixing"),
        cat.equilibrium_flame(name="flame"),
        cat.duct(LH, name="hot"),
        cat.choked_nozzle_outlet(A_THROAT, name="outlet"),
    ],
    edges=[(i, i + 1, AREA) for i in range(6)],
)

# Named edge handles (stable under with_params; topology is fixed).
E_AIR = net.edge_between("approach", "injector")
E_INJ_OUT = net.edge_between("injector", "mixing")
E_FLAME_IN = net.edge_between("mixing", "flame")
E_HOT = net.edge_between("flame", "hot")

sol = net.solve()
assert sol.converged, (sol.residual_norm, sol.print_residuals())
sol.verify()

u_mix = sol.edge(E_FLAME_IN)["u"]
tau = LM / u_mix
print(
    f"phi ~ {PHI:.2f}   T: {sol.edge(E_FLAME_IN)['T']:.0f} K "
    f"-> {sol.edge(E_HOT)['T']:.0f} K   chamber p = {sol.edge(E_HOT)['p'] / 1e5:.2f} bar"
)
print(f"mixing-duct velocity u = {u_mix:.1f} m/s  ->  transport lag tau = {tau * 1e3:.2f} ms")
print(f"Mach: {sol.edge(E_AIR)['M']:.3f} (cold) ... {sol.edge(E_HOT)['M']:.3f} (hot)")

## Anatomy of the loop: watching a wave make a mixture wave

Drive the inlet with a unit acoustic excitation and read the forced response along the chain.
Three checks against the derivation:

1. **Generation.** At the injector outlet, $\widehat{\phi}/\overline{\phi} = -\widehat{\dot m}_a/\overline{\dot m}_a$ (stiff fuel).
2. **Silence upstream.** No mixture wave rides the approach duct — composition is fixed at the air inlet.
3. **Transport.** Across the mixing duct, magnitude is conserved and the phase is $-\omega\tau$.

The same drive also yields the **emergent flame transfer function** $(\widehat{Q}/\overline{Q})/(\widehat{\dot m}_A/\overline{\dot m}) = \widehat{\dot m}_B/\widehat{\dot m}_A - e^{-\mathrm{i}\omega\tau}$, recovered from the linearized heat-release identity at the flame without attaching any dynamic source (the compact shape $1 - e^{-\mathrm{i}\omega\tau}$ is overlaid for reference).


In [ ]:
# Drive only the acoustic layer: copy the base, attach an anechoic driven inlet BC.
net_drv = net.copy()
net_drv.set("inlet", perturbation_bc=PerturbationBC.anechoic(driven=("acoustic",)))
sol_drv = net_drv.solve()
assert sol_drv.converged

fsw = np.linspace(20.0, 500.0, 121)
fr = sol_drv.forced_response(fsw, isentropic=False)

mdot_rel_inj = fr.field(E_INJ_OUT, "network")[:, 0] / MDOT  # mdot'/mdot_bar at injector outlet
mdot_rel_fl = fr.field(E_FLAME_IN, "network")[:, 0] / MDOT
mdot_a_rel = fr.field(E_AIR, "network")[:, 0] / MDOT_AIR
xi_up = fr.waves(E_AIR)[:, W_XI_FUEL]
xi_inj = fr.waves(E_INJ_OUT)[:, W_XI_FUEL]
xi_fl = fr.waves(E_FLAME_IN)[:, W_XI_FUEL]

# Equivalence-ratio amplitudes via phi = xi / ((1 - xi) FAR_st).
phi_rel_inj = xi_inj / (XI_FUEL * (1.0 - XI_FUEL))

# generation: stiff-fuel relation phi_hat/phi_bar = - mdot_a_hat / mdot_a_bar
gen_err = np.max(np.abs(phi_rel_inj + mdot_a_rel))
assert gen_err < 1e-5, f"generation relation violated: max |err| = {gen_err:.2e}"
print(f"generation: max |phi'/phi_bar + mdot_a'/mdot_a_bar| = {gen_err:.2e}")
print(f"upstream of the injector: max |xi_hat| = {np.abs(xi_up).max():.2e}  (silent, as derived)")

# transport: magnitude conserved, phase rotated by -omega*tau
phase = np.unwrap(np.angle(xi_fl / xi_inj))
slope_ms = -np.polyfit(2.0 * np.pi * fsw, phase, 1)[0] * 1e3
print(f"transport: fitted delay {slope_ms:.2f} ms  vs  L_m/u = {tau * 1e3:.2f} ms")
assert abs(slope_ms - tau * 1e3) < 0.02 * tau * 1e3

# Emergent FTF: heat-release identity at the flame, referenced to the injector face.
Q_rel = mdot_rel_fl + xi_fl / XI_FUEL
ftf = Q_rel / mdot_rel_inj
ftf_exact = mdot_rel_fl / mdot_rel_inj - np.exp(-1j * 2.0 * np.pi * fsw * tau)
ftf_compact = 1.0 - np.exp(-1j * 2.0 * np.pi * fsw * tau)
assert np.max(np.abs(ftf - ftf_exact)) < 1e-6

fig = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=(
        "generation: equivalence-ratio wave",
        "transport: phase across the mixing duct",
        "emergent FTF: gain",
        "emergent FTF: phase",
    ),
)
fig.add_scatter(
    x=fsw, y=np.abs(phi_rel_inj), name="|phi'/phi_bar| at injector", line=dict(color=COLORWAY[0]), row=1, col=1
)
fig.add_scatter(
    x=fsw,
    y=np.abs(mdot_a_rel),
    name="|mdot_a'|/mdot_a_bar (prediction)",
    line=dict(color=COLORWAY[1], dash="dash"),
    row=1,
    col=1,
)
fig.add_scatter(
    x=fsw,
    y=np.abs(xi_up) / (XI_FUEL * (1.0 - XI_FUEL)),
    name="|phi'| upstream (silent)",
    line=dict(color=COLORWAY[2]),
    row=1,
    col=1,
)
fig.add_scatter(x=fsw, y=phase, name="arg(xi'_B / xi'_A)", line=dict(color=COLORWAY[0]), row=1, col=2)
fig.add_scatter(
    x=fsw, y=-2.0 * np.pi * fsw * tau, name="-omega tau", line=dict(color=COLORWAY[1], dash="dash"), row=1, col=2
)
fig.add_scatter(x=fsw, y=np.abs(ftf), name="measured", line=dict(color=COLORWAY[0]), row=2, col=1)
fig.add_scatter(
    x=fsw,
    y=np.abs(ftf_exact),
    name="mdot_B/mdot_A - e^{-i omega tau}",
    line=dict(color=COLORWAY[1], dash="dash"),
    row=2,
    col=1,
)
fig.add_scatter(
    x=fsw,
    y=np.abs(ftf_compact),
    name="compact: |1 - e^{-i omega tau}|",
    line=dict(color=COLORWAY[2], dash="dot"),
    row=2,
    col=1,
)
fig.add_scatter(x=fsw, y=np.unwrap(np.angle(ftf)), name="measured", line=dict(color=COLORWAY[0]), row=2, col=2)
fig.add_scatter(
    x=fsw,
    y=np.unwrap(np.angle(ftf_exact)),
    name="exact prediction",
    line=dict(color=COLORWAY[1], dash="dash"),
    row=2,
    col=2,
)
fig.add_scatter(
    x=fsw,
    y=np.unwrap(np.angle(ftf_compact)),
    name="compact idealization",
    line=dict(color=COLORWAY[2], dash="dot"),
    row=2,
    col=2,
)
fig.update_xaxes(title_text="frequency [Hz]")
fig.update_yaxes(title_text="|phi'/phi_bar|", row=1, col=1)
fig.update_yaxes(title_text="phase [rad]", row=1, col=2)
fig.update_yaxes(title_text="|(Q'/Q_bar) / (mdot_A'/mdot_bar)|", row=2, col=1)
fig.update_yaxes(title_text="phase [rad]", row=2, col=2)
fig.update_layout(
    title="Equivalence-ratio wave and the emergent flame transfer function",
    width=900,
    height=650,
)
fig.show()
print(f"emergent FTF: max |measured - exact prediction| = {np.max(np.abs(ftf - ftf_exact)):.2e}")
print("compact idealization departs once the mixing duct is no longer acoustically short.")

## The spectrum: unstable, with the source block empty

No dynamic source is attached, yet the operator still carries unstable modes.
The spatial animation below shows the acoustic pressure of the most unstable mode; composition cannot be animated the same way (see the note in the next cell).


In [ ]:
modes = sol.eigenmodes(**EIG)
assert modes.certified, "the mode count could not be certified; widen the bands or densify the contour"

order = np.argsort(-modes.growth_rates)

display(modes)

fig_spectrum = modes.plot_spectrum()
fig_spectrum.update_layout(width=800, height=400)
fig_spectrum.show()

# Composition is not among the animate_mode variables.
# field_along_network / animate_mode reconstruct the duct field from the three
# acoustic/entropy characteristics (f, g, h) only — VARIABLE_SPEC is
# {p, u, rho, mdot, f, g, h}.  Stream-fraction (equivalence-ratio) waves live in
# the extended wave vector and are available at edge faces via modes.mode_waves /
# fr.waves, but they are not yet wired into the spatial reconstructors.
# Pressure is the natural acoustic readout of the loop; the composition carrier
# was verified in the forced-response cell above.
fig_anim_0 = modes.animate_mode(order[0], variable="p")
fig_anim_0.update_layout(width=800, height=400)
fig_anim_0.show()

## Which convected wave carries the loop?

The eigensolver can pin individual convected families to zero while keeping everything else — `convected="composition"` keeps the mixture-fraction waves and pins the entropy wave, `convected="entropy"` the reverse, and `isentropic=True` drops both.
Comparing the four spectra attributes the instability to its carrier:

* if the **composition** wave alone reproduces the unstable modes, the loop is the fuel-transport chain derived above;
* if the **entropy** wave alone did, the loop would instead be temperature spots shed by the flame's response to the local mass flux;
* with **both dropped**, the operator is the plain acoustic network, and whatever damping remains is the resonator's own.

A remark on the equilibrium flame: it conserves the total enthalpy through the burnt state, so the *temperature* perturbation of a richer-burnt parcel is encoded on the composition scalars — pinning the entropy wave does not remove the hot spot, only the part of it shed onto the entropy characteristic.

In [ ]:
assemblies = (
    ("full physics", dict(isentropic=False)),
    ("composition only", dict(convected="composition")),
    ("entropy only", dict(convected="entropy")),
    ("no convected waves", dict(isentropic=True)),
)
results = {}
for label, kw in assemblies:
    em = sol.eigenmodes(
        freq_band=EIG["freq_band"], growth_band=EIG["growth_band"], n_nodes=EIG["n_nodes"], n_probe=EIG["n_probe"], **kw
    )
    results[label] = em
    top = np.max(em.growth_rates) if em.n_modes else float("-inf")
    n_uns = int(np.sum(em.unstable)) if em.n_modes else 0
    print(
        f"{label:20s} certified={str(em.certified):5s}  modes={em.n_modes:2d}  "
        f"unstable={n_uns}  most unstable: {top:+.1f} 1/s"
    )

assert np.any(results["full physics"].unstable)
assert np.any(results["composition only"].unstable), "composition wave should carry the loop"
assert not np.any(results["entropy only"].unstable), "entropy wave alone should not destabilize"
assert not np.any(results["no convected waves"].unstable)

fig = go.Figure()
for i, (label, _) in enumerate(assemblies):
    em = results[label]
    fig.add_scatter(
        x=em.freqs,
        y=em.growth_rates,
        mode="markers",
        name=label,
        marker=dict(color=COLORWAY[i], size=9, symbol=["circle", "diamond", "square", "x"][i]),
    )
fig.add_hline(y=0.0, line=dict(color="grey", dash="dash"))
fig.update_xaxes(title_text="frequency [Hz]")
fig.update_yaxes(title_text="growth rate [1/s]")
fig.update_layout(title="The same operator, with convected families switched off one at a time", width=800, height=400)
fig.show()

The verdict is unambiguous.
The composition wave alone reproduces the unstable pair; the entropy wave alone leaves every mode damped; and the bare acoustic network keeps a single deeply damped resonance.
The unstable modes are not perturbed acoustic resonances at all — they belong to a family *created* by the convective loop, which is why they vanish with its carrier.

## Where the energy enters

For any mode the acoustic-power budget across the boundaries says who feeds it.
For the unstable mode the budget is dominated by the **outlet**: the arriving burnt inhomogeneity is accelerated through the choked nozzle and partly converted into sound, making the effective reflection stronger than energy-neutral — the indirect closure of the loop.
The flame's part in the loop is the quasi-steady conversion of the arriving mixture wave into that hot spot.

The direct (Rayleigh) closure is present too, but in this rig it loses: replacing the choked exit with an ideal open end removes the spot-to-sound transduction, and the same loop still *drives* — the full-physics damping is visibly smaller than the bare-acoustics damping — yet no longer beats the open end's radiation loss.

In [ ]:
best = int(np.argmax(modes.growth_rates))
print(modes.boundary_power(best))

# Open-end comparison: swapping the outlet *kind* (choked nozzle -> pressure outlet)
# is a structural change, not a scalar parameter, so it needs its own Network.
# Length, areas, BCs, and dynamic-source knobs would use with_params / set instead.
net_open = nefes.Network(
    nefes.equilibrium(),
    nodes=[
        cat.mass_flow_inlet(MDOT_AIR, 300.0, composition=AIR, basis="mole", name="inlet"),
        cat.duct(LA, name="approach"),
        cat.mass_source(MDOT_FUEL, 300.0, composition={"H2": 1.0}, basis="mole", name="injector"),
        cat.duct(LM, name="mixing"),
        cat.equilibrium_flame(name="flame"),
        cat.duct(LH, name="hot"),
        cat.pressure_outlet(
            1.0e5,
            Tt_backflow=300.0,
            composition=AIR,
            basis="mole",
            name="outlet",
            perturbation_bc=PerturbationBC.open_end(),
        ),
    ],
    edges=[(i, i + 1, AREA) for i in range(6)],
)
sol_open = net_open.solve()
assert sol_open.converged
for label, kw in (("full physics", dict(isentropic=False)), ("no convected waves", dict(isentropic=True))):
    em = sol_open.eigenmodes(
        freq_band=EIG["freq_band"],
        growth_band=EIG["growth_band"],
        n_nodes=EIG["n_nodes"],
        n_probe=EIG["n_probe"],
        **kw,
    )
    k = int(np.argmax(em.growth_rates))
    print(
        f"open outlet, {label:20s}: least-damped {em.freqs[k]:6.1f} Hz at {em.growth_rates[k]:+6.1f} 1/s "
        f"(certified={em.certified})"
    )
print("-> the loop contributes driving either way; the closed, choked resonator is where it wins.")

## The fingerprint: stability bands in the transport lag

The loop's phase is $\omega\tau$, so sweeping the mixing length $L_m$ (which changes $\tau$ and nothing else in the mean flow) must switch the instability on and off in bands, and the locked mode's frequency must fall as the lag grows.
This banding is the classical signature by which equivalence-ratio coupling is recognized in practice [2, 3].

The trajectory plot below tracks each eigenvalue in the (frequency, growth-rate) plane as $L_m$ is continued from a short mixing duct to a long one.
Read it as follows:

* Each coloured curve is one mode family, started at a seed (circle) and walked to the final $L_m$ (cross).
* Crossing the dashed line `growth = 0` is a stability switch: above it the mode is unstable.
* As $L_m$ grows, a family drifts **down in frequency** (longer lag → lower locked $\omega$ for a fixed $\omega\tau$ band) and eventually hands the unstable band to the next family — successive branches take over.
* Hover a point to read the mixing length at that sample.

That walk through the complex plane *is* the fingerprint: not a single resonance shifting, but a ladder of convective-loop modes trading the unstable band as the transport phase advances.


In [ ]:
# Seed from a shorter mixing duct (with_params leaves the base net untouched).
# eigenvalue_trajectory itself rebuilds each point via net.builder("mixing.length").
net_seed = net.with_params({"mixing.length": 0.075})
Lm_scan = np.linspace(0.075, 0.35, 23)
traj = net_seed.eigenvalue_trajectory(
    "mixing.length",
    Lm_scan,
    freq_band=EIG["freq_band"],
    growth_band=EIG["growth_band"],
    seed_kwargs=dict(n_nodes=EIG["n_nodes"], n_probe=EIG["n_probe"]),
)

fig_traj = traj.plot(title="Eigenvalue trajectories vs mixing length L_m")
fig_traj.update_layout(width=800, height=450)
fig_traj.show()

peak = max(float(np.nanmax(b.growth)) for b in traj.branches)
assert peak > 0.0
print(f"most unstable growth across the sweep: {peak:+.1f} 1/s")
print(
    "successive branches take over as L_m grows: each locks to the transport "
    "phase, drifts down in frequency, and hands the band to the next."
)

## Summary

* A steady fuel source, a mixing duct, and a static equilibrium flame — with the perturbation operator's source block empty — sustain a certified thermoacoustic instability.
  The dynamic response usually prescribed as a flame transfer function is here assembled from exact quasi-steady linearizations plus a convective delay, and recovers as $(\widehat{Q}/\overline{Q})/(\widehat{\dot m}_A/\overline{\dot m}) = \widehat{\dot m}_B/\widehat{\dot m}_A - e^{-\mathrm{i}\omega\tau}$ (and $1 - e^{-\mathrm{i}\omega\tau}$ in the compact limit).
* Each link was verified against the derivation: the stiff-fuel relation $\widehat{\phi}/\overline{\phi} = -\widehat{\dot m}_a/\overline{\dot m}_a$ holds to machine precision, the wave convects silently with the exact lag $L_m/\overline{u}$, and the flame turns it into a burnt hot spot.
* Switching convected families off one at a time attributes the loop to the **composition wave**; the entropy wave alone cannot sustain it here.
* The energy enters at the **choked nozzle** — the indirect closure — while the direct Rayleigh closure, though active, loses to radiation when the exit is opened.
* In the eigenvalue trajectory vs $L_m$, successive mode families cross into positive growth and drift down in frequency as the lag grows: the fingerprint of equivalence-ratio coupling.

Two caveats bound the physics.
First, the model convects the mixture wave without dispersion, while a real mixing duct smears the delay over a distribution of residence times; the sharp-delay result therefore overstates the driving at high frequency, and only the first few bands should be trusted quantitatively.
Second, if a *measured* flame transfer function is referenced to a velocity upstream of the fuel injection, it already contains this transport path — attaching such a response to a model that also resolves the feed line counts the mechanism twice.

**See also**: `equivalence_ratio_instability.ipynb` (the same chain driven by a prescribed fuel-flow response), `indirect_noise_instability.ipynb` (the entropy-wave analogue of the indirect closure), and `compositional_noise.ipynb` (the composition-to-sound conversion at a choked nozzle in isolation).

## References

1. J. J. Keller, "Thermoacoustic oscillations in combustion chambers of gas turbines," *AIAA Journal* 33 (12), 1995.
2. T. Lieuwen and B. T. Zinn, "The role of equivalence ratio oscillations in driving combustion instabilities in low NOx gas turbines," *Proceedings of the Combustion Institute* 27, 1998.
3. T. Sattelmayer, "Influence of the combustor aerodynamics on combustion instabilities from equivalence ratio fluctuations," *Journal of Engineering for Gas Turbines and Power* 125 (1), 2003.
4. W. Polifke, C. O. Paschereit and K. Döbbeling, "Constructive and destructive interference of acoustic and entropy waves in a premixed combustor with a choked exit," *International Journal of Acoustics and Vibration* 6 (3), 2001.
5. A. P. Dowling and S. R. Stow, "Acoustic analysis of gas turbine combustors," *Journal of Propulsion and Power* 19 (5), 2003.
6. E. Motheau, F. Nicoud and T. Poinsot, "Mixed acoustic-entropy combustion instabilities in gas turbines," *Journal of Fluid Mechanics* 749, 2014.


## Export for Nemo

`sol.to_yaml(path)` writes a UI-readable case embedding the converged mean flow; open it in Nemo to inspect the topology.

In [ ]:
import os
import tempfile

_out = os.path.join(tempfile.mkdtemp(), "fuel_transport_instability.yaml")
sol.to_yaml(_out)
print("saved case:", _out)